In [ ]:
"""
BDEW H25 – Stündliche Lastkurve
"""
 
import pandas as pd

 
# ── Konfiguration ─────────────────────────────────────────────────────────────
 
EXCEL_PATH  = "Rep2012_Repr_BDEW.xlsx"
SHEET_NAME  = "H25"
YEAR        = 2024
NORM_KWH    = 1_000_000  # Normierungswert laut Tabellenblatt
 
GEBAEUDE = pd.read_csv(r"C:\Users\JONATHANQ\Downloads\Gebäude-2981-heat.csv", index_col=0, decimal=',', sep=';')
 
# ── Dynamisierungsfunktion aus Excel ──────────────────────────────────────────
 
def dynamisierung(t):
    return -3.92e-10*t**4 + 3.20e-7*t**3 - 7.02e-5*t**2 + 2.10e-3*t + 1.24
 
# ── Tagestyp (ohne Feiertage) ─────────────────────────────────────────────────
 
def tagestyp(ts):
    wt = ts.weekday()
    if wt == 6: return "FT"   # Sonntag → Feiertagsprofil
    if wt == 5: return "SA"
    return "WT"
 
# ── SLP einlesen ─────────────────────────────────────────────────────────────
# Link zu BDEW: https://www.bdew.de/energie/standardlastprofile-strom/
raw = pd.read_csv("C:/Users/JONATHANQ/Downloads/NurH25_Repräsentative_Profile_BDEW_H25_G25_L25_P25_S25_Veröffentlichung.CSV", header=[2, 3], index_col=[0, 1], sep=";", encoding="latin-1", decimal=",")
 
MONATE = {
    "Januar":1,"Februar":2,"März":3,"April":4,"Mai":5,"Juni":6,
    "Juli":7,"August":8,"September":9,"Oktober":10,"November":11,"Dezember":12,
}
 
slp = {}
for (monat_str, tt), series in raw.items():
    monat_nr = MONATE.get(str(monat_str).split(".")[0].strip())
    tt = str(tt).strip().upper()
    if monat_nr and tt in ("SA", "FT", "WT"):
        for vs_idx, val in enumerate(series.dropna()):
            slp[(monat_nr, tt, vs_idx)] = float(val)
 
# ── Lastkurve berechnen ───────────────────────────────────────────────────────
 
def berechne_lastkurve(jahresverbrauch_kwh):
    skalierung = jahresverbrauch_kwh / NORM_KWH
    ts_index = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31 23:45", freq="15min")
 
    kwh_values = [
        slp.get((ts.month, tagestyp(ts), ts.hour * 4 + ts.minute // 15), 0.0)
        * dynamisierung(ts.day_of_year)
        * skalierung
        for ts in ts_index
    ]
 
    df_15min = pd.Series(kwh_values, index=ts_index)
    return df_15min.resample("h").sum().rename("kwh")
 
# ── Alle Gebäude berechnen ────────────────────────────────────────────────────
 
ergebnisse = {
    idx: berechne_lastkurve(row["Strombedarf (kWh/Jahr)"])
    for idx, row in GEBAEUDE.iterrows()
}
alle = pd.DataFrame(ergebnisse)
alle.index.name = "timestamp"
 
print(alle.head())
alle.to_csv("lastkurven.csv")


                     243687609  243687615  243687616  243687617  243687618  \
timestamp                                                                    
2024-01-01 00:00:00   1.024371        0.0   0.455276   0.910552   0.569095   
2024-01-01 01:00:00   0.880562        0.0   0.391361   0.782722   0.489201   
2024-01-01 02:00:00   0.833680        0.0   0.370524   0.741049   0.463156   
2024-01-01 03:00:00   0.826336        0.0   0.367260   0.734521   0.459075   
2024-01-01 04:00:00   0.862533        0.0   0.383348   0.766696   0.479185   

                     243687619  243687624  243687625  243687626  243687627  \
timestamp                                                                    
2024-01-01 00:00:00   0.455276   0.188930   0.455276   0.910552   0.317034   
2024-01-01 01:00:00   0.391361   0.162407   0.391361   0.782722   0.272527   
2024-01-01 02:00:00   0.370524   0.153760   0.370524   0.741049   0.258017   
2024-01-01 03:00:00   0.367260   0.152406   0.367260   0.734521